In [ ]:
import os
import pandas as pd
import numpy as np
from datetime import date
from sqlalchemy import create_engine, text
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# --------------------------------------------------
# LOAD DATA
# --------------------------------------------------
engine = create_engine(os.getenv("NBA_DATABASE_URL"))

df = pd.read_sql(
    text("SELECT * FROM fh_daily_ml_2023_onward_clean ORDER BY game_date"),
    engine
)

print(f"Loaded rows: {len(df)}")

# --------------------------------------------------
# DROP TIES (home_win = 0.5)
# --------------------------------------------------
df = df[df["home_win"].isin([0.0, 1.0])].copy()
print(f"Rows after dropping ties: {len(df)}")

# --------------------------------------------------
# FEATURES / TARGET
# --------------------------------------------------
feature_cols = [
    "home_r5_pts", "home_r5_allowed", "home_r5_margin", "home_wpct",
    "away_r5_pts", "away_r5_allowed", "away_r5_margin", "away_wpct",
    "home_plus_minus", "pct_margin"
]

X = df[feature_cols]
y = df["home_win"].astype(int)

# --------------------------------------------------
# TRAIN / TEST SPLIT (TIME-BASED)
# --------------------------------------------------
train_mask = df["game_date"] < date(2025, 1, 1)

X_train, X_test = X[train_mask], X[~train_mask]
y_train, y_test = y[train_mask], y[~train_mask]

print(f"Train rows: {len(X_train)} | Test rows: {len(X_test)}")

# --------------------------------------------------
# MODEL PIPELINE
# --------------------------------------------------
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        penalty="l2",
        C=1.0,
        solver="lbfgs",
        max_iter=1000
    ))
])

# --------------------------------------------------
# FIT
# --------------------------------------------------
pipe.fit(X_train, y_train)

# --------------------------------------------------
# EVALUATE
# --------------------------------------------------
proba_test = pipe.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= 0.5).astype(int)

auc = roc_auc_score(y_test, proba_test)
ll  = log_loss(y_test, proba_test)
bs  = brier_score_loss(y_test, proba_test)

print("\n📊 BASELINE LOGISTIC RESULTS (TEST SET)")
print(f"AUC        : {auc:.4f}")
print(f"Log Loss   : {ll:.4f}")
print(f"Brier Score: {bs:.4f}")

print("\nClassification Report (0.5 cutoff):")
print(classification_report(y_test, pred_test))

# --------------------------------------------------
# COEFFICIENTS
# --------------------------------------------------
coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coef": pipe.named_steps["logreg"].coef_[0]
}).sort_values("coef", ascending=False)

print("\n🔎 Model Coefficients (Higher = favors HOME):")
print(coef_df.to_string(index=False))



Loaded rows: 3264
Rows after dropping ties: 3159
Train rows: 1671 | Test rows: 1488

📊 BASELINE LOGISTIC RESULTS (TEST SET)
AUC        : 0.5846
Log Loss   : 0.6826
Brier Score: 0.2446

Classification Report (0.5 cutoff):
              precision    recall  f1-score   support

           0       0.56      0.45      0.50       696
           1       0.59      0.69      0.63       792

    accuracy                           0.58      1488
   macro avg       0.57      0.57      0.57      1488
weighted avg       0.57      0.58      0.57      1488


🔎 Model Coefficients (Higher = favors HOME):
        feature      coef
      home_wpct  0.242583
     pct_margin  0.158204
home_r5_allowed  0.056445
away_r5_allowed  0.045797
home_plus_minus  0.026340
    home_r5_pts  0.022971
 home_r5_margin -0.024476
    away_r5_pts -0.036424
 away_r5_margin -0.062886
      away_wpct -0.070696


In [ ]:
import os
import pandas as pd
import numpy as np
from datetime import date
from sqlalchemy import create_engine, text
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss

# --------------------------------------------------
# LOAD DATA
# --------------------------------------------------
engine = create_engine(
    os.getenv("NBA_DATABASE_URL")
)

df = pd.read_sql(
    text("SELECT * FROM fh_daily_ml_2023_onward_clean ORDER BY game_date"),
    engine
)

# Drop ties
df = df[df["home_win"].isin([0.0, 1.0])].copy()
df["home_win"] = df["home_win"].astype(int)

# --------------------------------------------------
# FEATURES
# --------------------------------------------------
feature_cols = [
    "home_r5_pts", "home_r5_allowed", "home_r5_margin", "home_wpct",
    "away_r5_pts", "away_r5_allowed", "away_r5_margin", "away_wpct",
    "home_plus_minus", "pct_margin"
]

X = df[feature_cols]
y = df["home_win"]

# --------------------------------------------------
# TRAIN / TEST SPLIT (TIME-BASED)
# --------------------------------------------------
train_mask = df["game_date"] < date(2025, 1, 1)

X_train, X_test = X[train_mask], X[~train_mask]
y_train, y_test = y[train_mask], y[~train_mask]

print(f"Train rows: {len(X_train)} | Test rows: {len(X_test)}")

# --------------------------------------------------
# XGBOOST MODEL (SHALLOW, STABLE)
# --------------------------------------------------
model = XGBClassifier(
    n_estimators=350,
    max_depth=3,
    learning_rate=0.06,
    subsample=0.85,
    colsample_bytree=0.85,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

# --------------------------------------------------
# FIT
# --------------------------------------------------
model.fit(X_train, y_train)

# --------------------------------------------------
# EVALUATE
# --------------------------------------------------
proba_test = model.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= 0.5).astype(int)

auc = roc_auc_score(y_test, proba_test)
ll  = log_loss(y_test, proba_test)
bs  = brier_score_loss(y_test, proba_test)

print("\n📊 XGBOOST RESULTS (TEST SET)")
print(f"AUC        : {auc:.4f}")
print(f"Log Loss   : {ll:.4f}")
print(f"Brier Score: {bs:.4f}")

# --------------------------------------------------
# FEATURE IMPORTANCE
# --------------------------------------------------
imp = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print("\n🔎 Feature Importance:")
print(imp.to_string(index=False))


Train rows: 1671 | Test rows: 1488

📊 XGBOOST RESULTS (TEST SET)
AUC        : 0.5482
Log Loss   : 0.7228
Brier Score: 0.2607

🔎 Feature Importance:
        feature  importance
home_plus_minus    0.139208
     pct_margin    0.126538
      home_wpct    0.104101
home_r5_allowed    0.097916
    home_r5_pts    0.093332
 home_r5_margin    0.090888
away_r5_allowed    0.090669
      away_wpct    0.089513
 away_r5_margin    0.084166
    away_r5_pts    0.083669


In [ ]:
import os
import pandas as pd
import numpy as np
from datetime import date
from sqlalchemy import create_engine, text

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss

# --------------------------------------------------
# LOAD DATA
# --------------------------------------------------
engine = create_engine(
    os.getenv("NBA_DATABASE_URL")
)

df = pd.read_sql(
    text("SELECT * FROM fh_daily_ml_2023_onward_clean ORDER BY game_date"),
    engine
)

# Drop ties
df = df[df["home_win"].isin([0.0, 1.0])].copy()
df["home_win"] = df["home_win"].astype(int)

# --------------------------------------------------
# FEATURES
# --------------------------------------------------
feature_cols = [
    "home_r5_pts", "home_r5_allowed", "home_r5_margin", "home_wpct",
    "away_r5_pts", "away_r5_allowed", "away_r5_margin", "away_wpct",
    "home_plus_minus", "pct_margin"
]

X = df[feature_cols]
y = df["home_win"]

# --------------------------------------------------
# TRAIN / TEST SPLIT (TIME-AWARE)
# --------------------------------------------------
train_mask = df["game_date"] < date(2025, 1, 1)

X_train, X_test = X[train_mask], X[~train_mask]
y_train, y_test = y[train_mask], y[~train_mask]

print(f"Train rows: {len(X_train)} | Test rows: {len(X_test)}")

# --------------------------------------------------
# BASE LOGISTIC PIPELINE
# --------------------------------------------------
base_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        penalty="l2",
        C=1.0,
        solver="lbfgs",
        max_iter=1000
    ))
])

# --------------------------------------------------
# CALIBRATION (PLATT / SIGMOID)
# --------------------------------------------------
calibrated_model = CalibratedClassifierCV(
    estimator=base_pipe,
    method="sigmoid",
    cv=5
)

# Fit ONLY on training data
calibrated_model.fit(X_train, y_train)

# --------------------------------------------------
# PREDICT ON TEST SET
# --------------------------------------------------
proba_test = calibrated_model.predict_proba(X_test)[:, 1]

# --------------------------------------------------
# METRICS AFTER CALIBRATION
# --------------------------------------------------
auc = roc_auc_score(y_test, proba_test)
ll  = log_loss(y_test, proba_test)
bs  = brier_score_loss(y_test, proba_test)

print("\n📊 CALIBRATED LOGISTIC RESULTS (TEST SET)")
print(f"AUC        : {auc:.4f}")
print(f"Log Loss   : {ll:.4f}")
print(f"Brier Score: {bs:.4f}")

# --------------------------------------------------
# BIN ANALYSIS
# --------------------------------------------------
bins = np.arange(0.40, 0.71, 0.02)  # focus on realistic betting range
df_bins = pd.DataFrame({
    "proba": proba_test,
    "actual": y_test.values
})

df_bins["bin"] = pd.cut(df_bins["proba"], bins=bins, include_lowest=True)

bin_summary = (
    df_bins
    .groupby("bin")
    .agg(
        games=("actual", "count"),
        win_rate=("actual", "mean"),
        avg_proba=("proba", "mean")
    )
    .reset_index()
)

# Edge vs fair probability
bin_summary["edge"] = bin_summary["win_rate"] - bin_summary["avg_proba"]

print("\n📈 BIN PERFORMANCE (TEST SET)")
print(bin_summary.to_string(index=False))


Train rows: 1671 | Test rows: 1488

📊 CALIBRATED LOGISTIC RESULTS (TEST SET)
AUC        : 0.5847
Log Loss   : 0.6816
Brier Score: 0.2441

📈 BIN PERFORMANCE (TEST SET)
          bin  games  win_rate  avg_proba      edge
(0.399, 0.42]     61  0.442623   0.408462  0.034161
 (0.42, 0.44]     86  0.441860   0.429223  0.012638
 (0.44, 0.46]    101  0.277228   0.450192 -0.172964
 (0.46, 0.48]     94  0.510638   0.470007  0.040632
  (0.48, 0.5]     75  0.546667   0.490367  0.056300
  (0.5, 0.52]     89  0.561798   0.509421  0.052377
 (0.52, 0.54]    112  0.491071   0.529679 -0.038608
 (0.54, 0.56]     91  0.626374   0.550241  0.076133
 (0.56, 0.58]    102  0.568627   0.570433 -0.001805
  (0.58, 0.6]    126  0.579365   0.589549 -0.010184
  (0.6, 0.62]    106  0.632075   0.610494  0.021581
 (0.62, 0.64]    101  0.574257   0.630739 -0.056482
 (0.64, 0.66]     85  0.694118   0.648732  0.045385
 (0.66, 0.68]     54  0.555556   0.668244 -0.112688
  (0.68, 0.7]     45  0.577778   0.688575 -0.110797


C:\Users\padre\AppData\Local\Temp\ipykernel_4752\997832617.py:105: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("bin")


In [ ]:
import os
import requests
import pandas as pd
import numpy as np
from datetime import datetime, date
from zoneinfo import ZoneInfo
from sqlalchemy import create_engine, text

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV

BALLDONTLIE_API_KEY = os.getenv("BALLDONTLIE_API_KEY")
DB_URL = os.getenv("NBA_DATABASE_URL")

PHX = ZoneInfo("America/Phoenix")
today_str = datetime.now(PHX).strftime("%Y-%m-%d")

CSV_PATH = rf"C:\Users\padre\MLB_Code_2024\nba_1st_half\daily_inference_{today_str}.csv"

CORE_MIN, CORE_MAX = 0.61, 0.63
LEAN_MIN, LEAN_MAX = 0.59, 0.61

engine = create_engine(DB_URL)
print(f"📅 Phoenix date: {today_str}")
print("🔌 Connected to Postgres")

# ----------------------------
# 1) TRAIN CALIBRATED LOGISTIC
# ----------------------------
train_df = pd.read_sql(
    text("SELECT * FROM fh_daily_ml_2023_onward_clean ORDER BY game_date"),
    engine
)

train_df = train_df[train_df["home_win"].isin([0.0, 1.0])].copy()
train_df["home_win"] = train_df["home_win"].astype(int)

feature_cols = [
    "home_r5_pts", "home_r5_allowed", "home_r5_margin", "home_wpct",
    "away_r5_pts", "away_r5_allowed", "away_r5_margin", "away_wpct",
    "home_plus_minus", "pct_margin"
]

X_all = train_df[feature_cols]
y_all = train_df["home_win"]

train_mask = train_df["game_date"] < date(2025, 1, 1)
X_train, y_train = X_all[train_mask], y_all[train_mask]

base_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(penalty="l2", C=1.0, solver="lbfgs", max_iter=1000))
])

cal_model = CalibratedClassifierCV(estimator=base_pipe, method="sigmoid", cv=5)

print(f"🧠 Training calibrated logistic on {len(X_train)} rows...")
cal_model.fit(X_train, y_train)
print("✅ Model trained")

# ----------------------------
# 2) TODAY'S SCHEDULE
# ----------------------------
print("🌐 Fetching today's NBA games...")
url = "https://api.balldontlie.io/v1/games"
params = {"dates[]": today_str, "per_page": 100}
headers = {"Authorization": BALLDONTLIE_API_KEY}

resp = requests.get(url, params=params, headers=headers, timeout=30)
resp.raise_for_status()
games = resp.json().get("data", [])
print(f"✅ Games found: {len(games)}")

if not games:
    print("No NBA games today.")
    raise SystemExit

schedule_rows = []
for g in games:
    home = g.get("home_team", {}) or {}
    away = g.get("visitor_team", {}) or {}
    schedule_rows.append({
        "game_id": g.get("id"),
        "status": g.get("status"),
        "home_team": home.get("abbreviation"),
        "away_team": away.get("abbreviation"),
        "start_utc": g.get("datetime"),
    })

schedule_df = pd.DataFrame(schedule_rows).sort_values("game_id")

# ----------------------------
# 3) LATEST SPLIT ROLLING-5 (incl wpct)
# ----------------------------
print("\n📊 Pulling latest split rolling-5 from fh_rolling_five_raw (incl wpct)...")

latest_sql = """
WITH latest AS (
    SELECT DISTINCT ON (team, split)
        team, split, date,
        rolling_5_pts, rolling_5_pts_allowed, rolling_5_margin, wpct
    FROM fh_rolling_five_raw
    WHERE rolling_5_margin IS NOT NULL
    ORDER BY team, split, date DESC
)
SELECT * FROM latest;
"""

roll_df = pd.read_sql(text(latest_sql), engine)
roll_df["date"] = pd.to_datetime(roll_df["date"])

home_roll = (
    roll_df[roll_df["split"] == "H"]
    .rename(columns={
        "date": "home_roll_date",
        "rolling_5_pts": "home_r5_pts",
        "rolling_5_pts_allowed": "home_r5_allowed",
        "rolling_5_margin": "home_r5_margin",
        "wpct": "home_wpct"
    })
    .drop(columns=["split"])
)

away_roll = (
    roll_df[roll_df["split"] == "A"]
    .rename(columns={
        "date": "away_roll_date",
        "rolling_5_pts": "away_r5_pts",
        "rolling_5_pts_allowed": "away_r5_allowed",
        "rolling_5_margin": "away_r5_margin",
        "wpct": "away_wpct"
    })
    .drop(columns=["split"])
)

out = (
    schedule_df
    .merge(home_roll, left_on="home_team", right_on="team", how="left")
    .drop(columns=["team"])
    .merge(away_roll, left_on="away_team", right_on="team", how="left")
    .drop(columns=["team"])
)

# Pre-game edges (same definition as training)
out["home_plus_minus"] = (out["home_r5_margin"] - out["away_r5_margin"]).round(2)
denom = out["home_r5_margin"].abs() + out["away_r5_margin"].abs()
out["pct_margin"] = np.where(denom.notna() & (denom != 0), out["home_plus_minus"] / denom, np.nan)
out["pct_margin"] = pd.Series(out["pct_margin"]).round(3)

# ✅ Ensure these exist on all rows (prevents KeyError)
out["home_win_proba"] = np.nan
out["signal"] = "NO DATA"

# Score only rows with complete features
out["has_features"] = out[feature_cols].notna().all(axis=1)
scored = out[out["has_features"]].copy()

if len(scored) > 0:
    scored["home_win_proba"] = cal_model.predict_proba(scored[feature_cols])[:, 1]
    scored["home_win_proba"] = scored["home_win_proba"].round(3)

    def flag(p):
        if CORE_MIN <= p <= CORE_MAX:
            return "CORE PLAY"
        if LEAN_MIN <= p < LEAN_MAX:
            return "LEAN"
        return "PASS"

    scored["signal"] = scored["home_win_proba"].apply(flag)

    # write back into out
    out.loc[scored.index, "home_win_proba"] = scored["home_win_proba"]
    out.loc[scored.index, "signal"] = scored["signal"]

# Final columns
final_cols = [
    "home_team", "away_team", "status", "start_utc",
    "home_r5_pts", "home_r5_allowed", "home_r5_margin", "home_wpct", "home_roll_date",
    "away_r5_pts", "away_r5_allowed", "away_r5_margin", "away_wpct", "away_roll_date",
    "home_plus_minus", "pct_margin",
    "home_win_proba", "signal"
]

final_df = out[final_cols].sort_values(["signal", "home_win_proba"], ascending=[True, False])

print("\n🏀 TODAY — CALIBRATED PROBABILITIES + SIGNALS")
print(final_df.to_string(index=False))

final_df.to_csv(CSV_PATH, index=False)
print(f"\n📁 CSV written to: {CSV_PATH}")



📅 Phoenix date: 2026-03-14
🔌 Connected to Postgres
🧠 Training calibrated logistic on 1671 rows...
✅ Model trained
🌐 Fetching today's NBA games...
✅ Games found: 7

📊 Pulling latest split rolling-5 from fh_rolling_five_raw (incl wpct)...

🏀 TODAY — CALIBRATED PROBABILITIES + SIGNALS
home_team away_team               status                start_utc  home_r5_pts  home_r5_allowed  home_r5_margin  home_wpct home_roll_date  away_r5_pts  away_r5_allowed  away_r5_margin  away_wpct away_roll_date  home_plus_minus  pct_margin  home_win_proba    signal
      PHI       BKN 2026-03-14T17:00:00Z 2026-03-14T17:00:00.000Z         60.0             62.8            -2.8        0.4     2026-03-10         52.6             62.8           -10.2        0.0     2026-03-12              7.4       0.569           0.621 CORE PLAY
      LAC       SAC 2026-03-15T02:30:00Z 2026-03-15T02:30:00.000Z         68.0             59.6             8.4        1.0     2026-03-13         59.0             66.2            -7.2    